# NeSLE A100 Benchmark + Native PPO

This notebook is the Colab path for the NeSLE A100 campaign. It clones the repo, mounts Drive, copies your legal Mario ROM from Drive, builds `nesle._cuda_core` for A100, verifies the CUDA import, then runs the benchmark commands.

Use a Colab GPU runtime. The target is **A100 80GB**. Results, checkpoints, and reports are written to Drive.

## What Must Be In Drive

Before running the notebook, put exactly this ROM file in Google Drive:

```text
MyDrive/mario_rl/roms/Super Mario Bros. (World).nes
```

The notebook creates these folders automatically:

```text
MyDrive/mario_rl/nesle_a100/
MyDrive/mario_rl/nesle_a100/checkpoints/
```

You do not need to upload snapshot files. The snapshot comes from the GitHub repo at `docs/data/smb_level1_1.state`.

In [ ]:
#@title Campaign Config
REPO_URL = "https://github.com/hbofz/Nesle-codex.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
PROJECT_SUBDIR = "project/mario-rl-ram"  #@param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/mario_rl"  #@param {type:"string"}
ROM_DRIVE_PATH = "/content/drive/MyDrive/mario_rl/roms/Super Mario Bros. (World).nes"  #@param {type:"string"}
RUN_NAME = "nesle_a100"  #@param {type:"string"}
CUDA_ARCH = "sm_80"  #@param ["sm_80", "sm_90"] {allow-input: true}
RUN_CORRECTNESS = True  #@param {type:"boolean"}
FULL_CAMPAIGN = False  #@param {type:"boolean"}

REPO_DIR = "/content/Nesle-codex"
PROJECT_DIR = f"{REPO_DIR}/{PROJECT_SUBDIR}"
OUTPUT_DIR = f"{DRIVE_ROOT}/{RUN_NAME}"
ROM_PROJECT_PATH = f"{PROJECT_DIR}/roms/Super Mario Bros. (World).nes"
SNAPSHOT_PATH = f"{REPO_DIR}/docs/data/smb_level1_1.state"

print("repo:", REPO_URL)
print("branch:", BRANCH)
print("project:", PROJECT_DIR)
print("ROM in Drive:", ROM_DRIVE_PATH)
print("output:", OUTPUT_DIR)

## 1. Mount Drive, Clone Repo, And Define Helpers

If the repo is private, add a Colab secret named `GITHUB_TOKEN` with read access before running this cell.

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import shutil
import subprocess
import sys


def run(cmd, cwd=None, check=True, env=None):
    print("$", " ".join(cmd) if isinstance(cmd, list) else cmd)
    completed = subprocess.run(
        cmd,
        cwd=cwd,
        shell=isinstance(cmd, str),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        env=env,
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if check and completed.returncode != 0:
        raise RuntimeError(f"command failed with code {completed.returncode}: {cmd}")
    return completed


def prepend_env_path(env, key, value):
    old = env.get(key)
    env[key] = value if not old else f"{value}{os.pathsep}{old}"


def find_nvcc():
    candidates = [
        shutil.which("nvcc"),
        "/usr/local/cuda/bin/nvcc",
        "/usr/local/cuda-12.8/bin/nvcc",
        "/usr/local/cuda-12.6/bin/nvcc",
        "/usr/local/cuda-12.5/bin/nvcc",
        "/usr/local/cuda-12.4/bin/nvcc",
    ]
    for candidate in candidates:
        if candidate and Path(candidate).exists():
            return candidate
    for root in sys.path:
        for candidate in Path(root).glob("nvidia/cuda_nvcc/bin/nvcc"):
            if candidate.exists():
                return str(candidate)
    return None


def ensure_nvcc():
    nvcc = find_nvcc()
    if nvcc:
        return nvcc

    print("nvcc was not found. Trying the lightweight pip nvcc package first.")
    run([sys.executable, "-m", "pip", "install", "nvidia-cuda-nvcc-cu12"])
    nvcc = find_nvcc()
    if nvcc:
        prepend_env_path(os.environ, "PATH", str(Path(nvcc).parent))
        return nvcc

    print("pip nvcc was not enough. Trying Colab apt CUDA nvcc packages.")
    run("apt-get update -qq", check=False)
    run("apt-get install -y cuda-nvcc-12-5 || apt-get install -y cuda-nvcc-12-4 || apt-get install -y cuda-toolkit-12-5", check=False)
    nvcc = find_nvcc()
    if nvcc:
        prepend_env_path(os.environ, "PATH", str(Path(nvcc).parent))
        return nvcc

    raise RuntimeError("nvcc is still missing. In Colab, switch to a GPU runtime with CUDA toolkit support, then rerun from this cell.")


drive.mount('/content/drive')
Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

clone_url = REPO_URL
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
if token and clone_url.startswith('https://github.com/'):
    clone_url = clone_url.replace('https://github.com/', f'https://{token}@github.com/', 1)

if Path(REPO_DIR).exists():
    run(['git', 'fetch', 'origin'], cwd=REPO_DIR)
    run(['git', 'checkout', BRANCH], cwd=REPO_DIR)
    run(['git', 'pull', '--ff-only'], cwd=REPO_DIR)
else:
    run(['git', 'clone', '--branch', BRANCH, clone_url, REPO_DIR])

run(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR)
print('project exists:', Path(PROJECT_DIR).is_dir())

## 2. Install Minimal Packages And Copy ROM

This installs NeSLE and the Mario benchmark harness in editable mode. The Mario harness is installed with `--no-deps` because this notebook uses the native NeSLE path, not Stable-Retro.

In [ ]:
run([sys.executable, '-m', 'pip', 'install', '-U', 'pip', 'setuptools', 'wheel', 'pybind11', 'numpy'])
run([sys.executable, '-m', 'pip', 'install', '-e', REPO_DIR])
run([sys.executable, '-m', 'pip', 'install', '-e', PROJECT_DIR, '--no-deps'])

Path(f'{PROJECT_DIR}/roms').mkdir(parents=True, exist_ok=True)
if not Path(ROM_DRIVE_PATH).is_file():
    raise FileNotFoundError(
        'ROM not found. Put your legal ROM here exactly: '        f'{ROM_DRIVE_PATH}'
    )
shutil.copy2(ROM_DRIVE_PATH, ROM_PROJECT_PATH)
print('ROM copied to:', ROM_PROJECT_PATH)

print('GPU:')
run('nvidia-smi', check=False)

print('PyTorch CUDA:')
run([sys.executable, '-c', 'import torch; print(torch.__version__); print(torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no cuda")'])

## 3. Build And Verify `nesle._cuda_core`

This is the important cell. It checks for `nvcc`, builds `_cuda_core` into the repo, imports it, and only then runs the benchmark preflight.

In [ ]:
nvcc = ensure_nvcc()
print('nvcc:', nvcc)
run([nvcc, '--version'])

BENCH_ENV = os.environ.copy()
prepend_env_path(BENCH_ENV, 'PYTHONPATH', f'{REPO_DIR}/src')
prepend_env_path(BENCH_ENV, 'PYTHONPATH', f'{PROJECT_DIR}/src')
prepend_env_path(BENCH_ENV, 'PATH', str(Path(nvcc).parent))
BENCH_ENV['NESLE_CUDA_ARCH'] = CUDA_ARCH
BENCH_ENV['NESLE_REQUIRE_CUDA'] = '1'
BENCH_ENV['PYTHON'] = sys.executable

run(['sh', 'scripts/build_cuda_extension.sh'], cwd=REPO_DIR, env=BENCH_ENV)
run([
    sys.executable,
    '-c',
    'import nesle, nesle._cuda_core as c; print(nesle.__file__); print(c.__file__); print(c.CudaBatch(1,4).name)',
], env=BENCH_ENV)

preflight_cmd = [
    sys.executable, '-m', 'mario_rl.nesle_bench', 'preflight',
    '--cuda-arch', CUDA_ARCH,
    '--nesle-root', REPO_DIR,
    '--rom', ROM_PROJECT_PATH,
    '--snapshot', SNAPSHOT_PATH,
    '--output-dir', OUTPUT_DIR,
]
if RUN_CORRECTNESS:
    preflight_cmd.append('--run-correctness')
run(preflight_cmd, cwd=PROJECT_DIR, env=BENCH_ENV)

## 4. Env-Only Throughput Sweep

Raw `CudaBatch` throughput with snapshot reset, no render copy, and no observation copy.

In [ ]:
#@title Env Sweep Options
ENV_COUNTS = "1,8,32,128,512,1024,2048,4096,8192,16384,32768,65536,98304,131072"  #@param {type:"string"}
ENV_WARMUP_STEPS = 30  #@param {type:"integer"}
ENV_TIMED_STEPS = 200  #@param {type:"integer"}

env_cmd = [
    sys.executable, '-m', 'mario_rl.nesle_bench', 'env-sweep',
    '--nesle-root', REPO_DIR,
    '--rom', ROM_PROJECT_PATH,
    '--snapshot', SNAPSHOT_PATH,
    '--output-dir', OUTPUT_DIR,
    '--env-counts', ENV_COUNTS,
    '--warmup-steps', str(ENV_WARMUP_STEPS),
    '--timed-steps', str(ENV_TIMED_STEPS),
]
run(env_cmd, cwd=PROJECT_DIR, env=BENCH_ENV)

## 5. Native PPO Short Sweep

Run this after preflight passes. Start with short PPO updates, then use the stress cell once you know a stable env count.

In [ ]:
#@title PPO Sweep Options
PPO_COUNTS = "1024,4096,8192,16384,32768,65536"  #@param {type:"string"}
PPO_UPDATES = 2  #@param {type:"integer"}
N_STEPS = 128  #@param {type:"integer"}
BATCH_SIZE = 8192  #@param {type:"integer"}
HIDDEN_SIZE = 256  #@param {type:"integer"}

ppo_cmd = [
    sys.executable, '-m', 'mario_rl.nesle_bench', 'ppo-sweep',
    '--nesle-root', REPO_DIR,
    '--rom', ROM_PROJECT_PATH,
    '--snapshot', SNAPSHOT_PATH,
    '--output-dir', OUTPUT_DIR,
    '--ppo-env-counts', PPO_COUNTS,
    '--ppo-updates', str(PPO_UPDATES),
    '--n-steps', str(N_STEPS),
    '--batch-size', str(BATCH_SIZE),
    '--hidden-size', str(HIDDEN_SIZE),
    '--ppo-action-space', 'mario',
]
run(ppo_cmd, cwd=PROJECT_DIR, env=BENCH_ENV)

## 6. Long Stress Run

Use this after the short sweeps identify a stable env count. The default target is the first serious A100 run.

In [ ]:
#@title Stress Options
STRESS_ENVS = 65536  #@param {type:"integer"}
STRESS_TIMESTEPS = 75000000  #@param {type:"integer"}

if FULL_CAMPAIGN:
    cmd = [
        sys.executable, '-m', 'mario_rl.nesle_bench', 'all',
        '--nesle-root', REPO_DIR,
        '--rom', ROM_PROJECT_PATH,
        '--snapshot', SNAPSHOT_PATH,
        '--output-dir', OUTPUT_DIR,
        '--env-counts', ENV_COUNTS,
        '--ppo-env-counts', PPO_COUNTS,
        '--ppo-action-space', 'mario',
        '--n-steps', str(N_STEPS),
        '--batch-size', str(BATCH_SIZE),
        '--hidden-size', str(HIDDEN_SIZE),
    ]
else:
    cmd = [
        sys.executable, '-m', 'mario_rl.nesle_bench', 'stress',
        '--nesle-root', REPO_DIR,
        '--rom', ROM_PROJECT_PATH,
        '--snapshot', SNAPSHOT_PATH,
        '--output-dir', OUTPUT_DIR,
        '--stress-envs', str(STRESS_ENVS),
        '--stress-timesteps', str(STRESS_TIMESTEPS),
        '--ppo-action-space', 'mario',
        '--n-steps', str(N_STEPS),
        '--batch-size', str(BATCH_SIZE),
        '--hidden-size', str(HIDDEN_SIZE),
    ]
run(cmd, cwd=PROJECT_DIR, env=BENCH_ENV)

## 7. Inspect Outputs

Open the generated report and list all artifacts written to Drive.

In [ ]:
from IPython.display import Markdown, display

report = Path(OUTPUT_DIR) / 'nesle_a100_report.md'
if report.exists():
    display(Markdown(report.read_text()))
else:
    print('No report yet:', report)

print('\nArtifacts:')
for path in sorted(Path(OUTPUT_DIR).rglob('*')):
    if path.is_file():
        print(path)
